In [7]:
import os
print(os.listdir('.'))


['.config', 'uber_fares.csv', 'sample_data']


In [8]:
with open('uber_fares.csv', 'r') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4: # Print first 5 lines
            break


In [9]:
!pip install -q kagglehub

import kagglehub
import pandas as pd
import os

# Download the dataset using kagglehub
path = kagglehub.dataset_download("yasserh/uber-fares-dataset")

print("Path to dataset files:", path)

# Inspect the downloaded files to confirm the CSV name
dataset_files = os.listdir(path)
print("Files in dataset directory:", dataset_files)

# Assuming the CSV file is named 'uber.csv' or similar within the downloaded path
# Adjust this if inspection shows a different filename
csv_file_name = None
for file in dataset_files:
    if file.endswith('.csv'):
        csv_file_name = file
        break

if csv_file_name:
    df = pd.read_csv(os.path.join(path, csv_file_name))
    print("Successfully loaded data. Displaying head:")
    print(df.head())
else:
    print("No CSV file found in the downloaded dataset.")


100%|██████████| 7.04M/7.04M [00:00<00:00, 9.39MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/yasserh/uber-fares-dataset/versions/1
Files in dataset directory: ['uber.csv']
Successfully loaded data. Displaying head:
   Unnamed: 0                            key  fare_amount  \
0    24238194    2015-05-07 19:52:06.0000003          7.5   
1    27835199    2009-07-17 20:04:56.0000002          7.7   
2    44984355   2009-08-24 21:45:00.00000061         12.9   
3    25894730    2009-06-26 08:22:21.0000001          5.3   
4    17610152  2014-08-28 17:47:00.000000188         16.0   

           pickup_datetime  pickup_longitude  pickup_latitude  \
0  2015-05-07 19:52:06 UTC        -73.999817        40.738354   
1  2009-07-17 20:04:56 UTC        -73.994355        40.728225   
2  2009-08-24 21:45:00 UTC        -74.005043        40.740770   
3  2009-06-26 08:22:21 UTC        -73.976124        40.790844   
4  2014-08-28 17:47:00 UTC        -73.925023        40.744085   

   dropoff_longitude  dropoff_latitude  passenger_count  
0      

In [10]:
print("DataFrame Info:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDescriptive Statistics:")
print(df.describe())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Unnamed: 0         200000 non-null  int64  
 1   key                200000 non-null  object 
 2   fare_amount        200000 non-null  float64
 3   pickup_datetime    200000 non-null  object 
 4   pickup_longitude   200000 non-null  float64
 5   pickup_latitude    200000 non-null  float64
 6   dropoff_longitude  199999 non-null  float64
 7   dropoff_latitude   199999 non-null  float64
 8   passenger_count    200000 non-null  int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 13.7+ MB

Missing Values:
Unnamed: 0           0
key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    1
dropoff_latitude     1
passenger_count      0
dtype: int64

Descriptive Statistics:
         U

In [11]:
print("Original DataFrame shape:", df.shape)

# Drop rows with missing dropoff latitude or longitude
df.dropna(subset=['dropoff_longitude', 'dropoff_latitude'], inplace=True)
print("DataFrame shape after dropping missing values:", df.shape)

# Remove rows with fare_amount less than or equal to 0 (unrealistic)
df = df[df['fare_amount'] > 0]

# Remove rows with passenger_count less than 1 or greater than a reasonable maximum (e.g., 6 for a standard taxi)
df = df[df['passenger_count'] >= 1]
df = df[df['passenger_count'] <= 6]

# Define valid geographical coordinate ranges
# Latitude: -90 to 90, Longitude: -180 to 180
df = df[df['pickup_latitude'].between(-90, 90)]
df = df[df['dropoff_latitude'].between(-90, 90)]
df = df[df['pickup_longitude'].between(-180, 180)]
df = df[df['dropoff_longitude'].between(-180, 180)]

print("DataFrame shape after cleaning invalid data:", df.shape)
print("Descriptive Statistics after cleaning:")
print(df.describe())

Original DataFrame shape: (200000, 9)
DataFrame shape after dropping missing values: (199999, 9)
DataFrame shape after cleaning invalid data: (199256, 9)
Descriptive Statistics after cleaning:
         Unnamed: 0    fare_amount  pickup_longitude  pickup_latitude  \
count  1.992560e+05  199256.000000     199256.000000    199256.000000   
mean   2.771597e+07      11.369376        -72.504173        39.919172   
std    1.601417e+07       9.905986         10.442243         6.127757   
min    1.000000e+00       0.010000        -93.824668       -74.015515   
25%    1.382668e+07       6.000000        -73.992063        40.734794   
50%    2.775602e+07       8.500000        -73.981825        40.752582   
75%    4.156047e+07      12.500000        -73.967162        40.767155   
max    5.542357e+07     499.000000         40.808425        48.018760   

       dropoff_longitude  dropoff_latitude  passenger_count  
count      199256.000000     199256.000000    199256.000000  
mean          -72.514408 

In [12]:
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
print("DataFrame Info after datetime conversion:")
df.info()

DataFrame Info after datetime conversion:
<class 'pandas.core.frame.DataFrame'>
Index: 199256 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   Unnamed: 0         199256 non-null  int64              
 1   key                199256 non-null  object             
 2   fare_amount        199256 non-null  float64            
 3   pickup_datetime    199256 non-null  datetime64[ns, UTC]
 4   pickup_longitude   199256 non-null  float64            
 5   pickup_latitude    199256 non-null  float64            
 6   dropoff_longitude  199256 non-null  float64            
 7   dropoff_latitude   199256 non-null  float64            
 8   passenger_count    199256 non-null  int64              
dtypes: datetime64[ns, UTC](1), float64(5), int64(2), object(1)
memory usage: 15.2+ MB


In [13]:
import numpy as np
from math import radians, sin, cos, atan2, sqrt

def haversine_distance(pickup_latitude, pickup_longitude, dropoff_latitude, dropoff_longitude):
    R = 6371  # Earth's radius in kilometers

    lat1 = radians(pickup_latitude)
    lon1 = radians(pickup_longitude)
    lat2 = radians(dropoff_latitude)
    lon2 = radians(dropoff_longitude)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

df['distance'] = df.apply(lambda row: haversine_distance(
    row['pickup_latitude'],
    row['pickup_longitude'],
    row['dropoff_latitude'],
    row['dropoff_longitude']
), axis=1)

print("DataFrame head with new 'distance' column:")
print(df.head())

DataFrame head with new 'distance' column:
   Unnamed: 0                            key  fare_amount  \
0    24238194    2015-05-07 19:52:06.0000003          7.5   
1    27835199    2009-07-17 20:04:56.0000002          7.7   
2    44984355   2009-08-24 21:45:00.00000061         12.9   
3    25894730    2009-06-26 08:22:21.0000001          5.3   
4    17610152  2014-08-28 17:47:00.000000188         16.0   

            pickup_datetime  pickup_longitude  pickup_latitude  \
0 2015-05-07 19:52:06+00:00        -73.999817        40.738354   
1 2009-07-17 20:04:56+00:00        -73.994355        40.728225   
2 2009-08-24 21:45:00+00:00        -74.005043        40.740770   
3 2009-06-26 08:22:21+00:00        -73.976124        40.790844   
4 2014-08-28 17:47:00+00:00        -73.925023        40.744085   

   dropoff_longitude  dropoff_latitude  passenger_count  distance  
0         -73.999512         40.723217                1  1.683323  
1         -73.994710         40.750325                1  

In [14]:
df['year'] = df['pickup_datetime'].dt.year
df['month'] = df['pickup_datetime'].dt.month
df['day'] = df['pickup_datetime'].dt.day
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek

print("DataFrame head with new time-based features:")
print(df.head())
print("\nDataFrame Info after adding time-based features:")
df.info()

DataFrame head with new time-based features:
   Unnamed: 0                            key  fare_amount  \
0    24238194    2015-05-07 19:52:06.0000003          7.5   
1    27835199    2009-07-17 20:04:56.0000002          7.7   
2    44984355   2009-08-24 21:45:00.00000061         12.9   
3    25894730    2009-06-26 08:22:21.0000001          5.3   
4    17610152  2014-08-28 17:47:00.000000188         16.0   

            pickup_datetime  pickup_longitude  pickup_latitude  \
0 2015-05-07 19:52:06+00:00        -73.999817        40.738354   
1 2009-07-17 20:04:56+00:00        -73.994355        40.728225   
2 2009-08-24 21:45:00+00:00        -74.005043        40.740770   
3 2009-06-26 08:22:21+00:00        -73.976124        40.790844   
4 2014-08-28 17:47:00+00:00        -73.925023        40.744085   

   dropoff_longitude  dropoff_latitude  passenger_count  distance  year  \
0         -73.999512         40.723217                1  1.683323  2015   
1         -73.994710         40.750325   

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Define features (X) and target (y)
X = df[['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
        'passenger_count', 'distance', 'year', 'month', 'day', 'hour', 'day_of_week']]
y = df['fare_amount']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")
print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

Linear Regression model trained successfully.
Training set shape: (159404, 11), (159404,)
Test set shape: (39852, 11), (39852,)


In [16]:
from sklearn.ensemble import RandomForestRegressor

# Initialize the Random Forest Regressor model
random_forest_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Fit the model to the training data
random_forest_model.fit(X_train, y_train)

print("Random Forest Regressor model trained successfully.")

Random Forest Regressor model trained successfully.


In [17]:
import xgboost as xgb

# Initialize the XGBoost Regressor model
# Using some common parameters for a baseline model
xgboost_model = xgb.XGBRegressor(
    objective='reg:squarederror', # Objective function for regression
    n_estimators=100,             # Number of boosting rounds
    learning_rate=0.1,            # Step size shrinkage to prevent overfitting
    max_depth=5,                  # Maximum depth of a tree
    subsample=0.8,                # Subsample ratio of the training instance
    colsample_bytree=0.8,         # Subsample ratio of columns when constructing each tree
    random_state=42               # Random seed for reproducibility
)

# Fit the model to the training data
xgboost_model.fit(X_train, y_train)

print("XGBoost Regressor model trained successfully.")

XGBoost Regressor model trained successfully.


In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Make predictions with the Linear Regression model
y_pred_lr = linear_model.predict(X_test)

# Evaluate Linear Regression model
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Model Performance:")
print(f"  MAE: {mae_lr:.4f}")
print(f"  MSE: {mse_lr:.4f}")
print(f"  RMSE: {rmse_lr:.4f}")
print(f"  R-squared: {r2_lr:.4f}")

Linear Regression Model Performance:
  MAE: 5.9882
  MSE: 95.3461
  RMSE: 9.7645
  R-squared: 0.0174


In [19]:
y_pred_rf = random_forest_model.predict(X_test)

# Evaluate Random Forest Regressor model
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Regressor Model Performance:")
print(f"  MAE: {mae_rf:.4f}")
print(f"  MSE: {mse_rf:.4f}")
print(f"  RMSE: {rmse_rf:.4f}")
print(f"  R-squared: {r2_rf:.4f}")

Random Forest Regressor Model Performance:
  MAE: 1.9658
  MSE: 19.6430
  RMSE: 4.4320
  R-squared: 0.7976


In [20]:
y_pred_xgb = xgboost_model.predict(X_test)

# Evaluate XGBoost Regressor model
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost Regressor Model Performance:")
print(f"  MAE: {mae_xgb:.4f}")
print(f"  MSE: {mse_xgb:.4f}")
print(f"  RMSE: {rmse_xgb:.4f}")
print(f"  R-squared: {r2_xgb:.4f}")

XGBoost Regressor Model Performance:
  MAE: 1.9873
  MSE: 18.9903
  RMSE: 4.3578
  R-squared: 0.8043
